In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_kama_extfile(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #     "ExtKAMAId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "ExtKAMAKey string,"
        #     "BranchCode string,"
        #     "ExposureKey string,"
        #     "AgreementKey string,"
        #     "CDI string,"
        #     "PartyName string,"
        #     "PartyAddress string,"
        #     "PartyCity string,"
        #     "PartyZipCode string,"
        #     "PartyTaxRegistrationNumber string,"
        #     "PartyIDNumber string,"
        #     "PartyBirthdate string,"
        #     "PartyOccupation string,"
        #     "PartyPhone string,"
        #     "StatementAddress string,"
        #     "StatementCity string,"
        #     "StatementTK string,"
        #     "GuarantorName string,"
        #     "GuarantorAddress string,"
        #     "GuarantorCity string,"
        #     "GuarantorTK string,"
        #     "GuarantorAFM string,"
        #     "GuarantorIDNumber string,"
        #     "GuarantorPhone string,"
        #     "GuarantorBirthdate string,"
        #     "GuarantorOccupation string,"
        #     "AgreementAmount string,"
        #     "AgreementDate string,"
        #     "AgreementUpdateDate string,"
        #     "OpenExposureDate string,"
        #     "LegalFrame string,"
        #     "LegalFrameFlag string,"
        #     "ExposureStatus string,"
        #     "InstallmentPercentage string,"
        #     "InterestRate string,"
        #     "InterestRateType string,"
        #     "InterestRateBasis string,"
        #     "InterestRateCategory string,"
        #     "InterestExpiredDate string,"
        #     "InterestRateSpread string,"
        #     "InterestRateSpreadSign string,"
        #     "AmountApproved string,"
        #     "AmountDisbursed string,"
        #     "FirstDisbursementDate string,"
        #     "TotalPayments string,"
        #     "TotalBalance string,"
        #     "Bucket0 string,"
        #     "Bucket1 string,"
        #     "Bucket2 string,"
        #     "Bucket3 string,"
        #     "Bucket4 string,"
        #     "Bucket5 string,"
        #     "Bucket6 string,"
        #     "Bucket7Plus string,"
        #     "DelinquentAmount string,"
        #     "InstallmentAmount string,"
        #     "TotalInstallments string,"
        #     "DelinquentInstallments string,"
        #     "BucketMaxDate string,"
        #     "DaysInDelinquency string,"
        #     "LastPayDate string,"
        #     "LastPayAmount string,"
        #     "NextDueDate string,"
        #     "ExposureDuration string,"
        #     "GracePeriodFlag string,"
        #     "GracePeriodEndDate string,"
        #     "ServiceWay string,"
        #     "ProductCode string,"
        #     "PrenotationFlag string,"
        #     "AssignmentDate string,"
        #     "AssignmentFlag string,"
        #     "ExceptionReason string,"
        #     "DisbursementLastDate string,"
        #     "BalanceBilling string,"
        #     "BalanceBucket0Billing string,"
        #     "PartyGender string,"
        #     "BucketDME string,"
        #     "InstallmentsFrequency string,"
        #     "Proorismos string,"
        #     "DeferredCum string,"
        #     "ApplicationCode string,"
        #     "PartyCategory string,"
        #     "DebitExposureKey string,"
        #     "Party2Name string,"
        #     "Party2Phone string,"
        #     "Party3Name string,"
        #     "Party3Phone string,"
        #     "Party4Name string,"
        #     "Party4Phone string,"
        #     "CommercialValue string,"
        #     "ObjectiveValue string,"
        #     "LienOrder string,"
        #     "LienAmount string,"
        #     "RealEstateCategory string,"
        #     "NaceCode string,"
        #     "Country string,"
        #     "Nationality string,"
        #     "RealEstateCollateral string,"
        #     "RealEstateAmount string,"
        #     "OnBalanceRealEstateAmount string,"
        #     "SharesCollateral string,"
        #     "SharesAmount string,"
        #     "OnBalanceSharesAmount string,"
        #     "LetterOfCreditCollateral string,"
        #     "LetterOfCreditAmount string,"
        #     "OnBalanceLetterOfCreditAmount string,"
        #     "LetterOfGuranteeCollateral string,"
        #     "LetterOfGuranteeAmounts string,"
        #     "OnBalanceLetterOfGuaranteeAmount string,"
        #     "LetterOfPublicGuaranteeCollateral string,"
        #     "LetterOfPublicGuaranteeAmount string,"
        #     "OnBalanceLetterOfPublicGuaranteeAmount string,"
        #     "OtherOfGuaranteesCollateral string,"
        #     "OtherOfGuaranteesAmount string,"
        #     "OnBalanceOtherOfGuaranteesAmount string,"
        #     "GoodsCollateral string,"
        #     "GoodsAmount string,"
        #     "OnBalanceGoodsAmount string,"
        #     "ArchiveFilesCollateral string,"
        #     "ArchiveFilesAmount string,"
        #     "OnBalanceArchiveFilesAmount string,"
        #     "NewTypeCollateral string,"
        #     "NewTypeAmount string,"
        #     "OnBalanceNewTypeAmount string,"
        #     "ExposureInfo string,"
        #     "NextFinalInterest string,"
        #     "NextInerestType string,"
        #     "NextBaseRate string,"
        #     "NextInterest string,"
        #     "RealEstateKey string,"
        #     "Bucket7 string,"
        #     "Bucket8 string,"
        #     "Bucket9 string,"
        #     "Bucket10 string,"
        #     "Bucket11 string,"
        #     "Bucket12 string,"
        #     "Bucket13 string,"
        #     "DMESettlement string,"
        #     "DMERestructuring string,"
        #     "DMELetterOfPublicGurantee string,"
        #     "AddApplicationKey string,"
        #     "RealEstateUsage string,"
        #     "ExposureStatusChangeDate string,"
        #     "PerformingPartyUnitCode string,"
        #     "PerformingPartyUnit string,"
        #     "AnnouncedPartyUnitCode string,"
        #     "AnnouncedPartyUnit string,"
        #     "DueCapital string,"
        #     "DueInterest string,"
        #     "DueExpense string,"
        #     "CapitalisationOfInterest string,"
        #     "SubsidyByPublic string,"
        #     "DaysPastDueUnrealisedInterest string,"
        #     "CommercialFlag string,"
        #     "UnrealisedInterestBefore31032013 string,"
        #     "EmporikhAgreementCode string,"
        #     "BucketNCKAG string,"
        #     "MinBucketNCKAG string,"
        #     "LastRestructuringDate string,"
        #     "ExposureExpiredDate string,"
        #     "InitialExposureExpiredDate string,"
        #     "SigloProductCode string,"
        #     "AbhostExposure string,"
        #     "BU string,"
        #     "AbhostAgreement string,"
        #     "Field165 string,"
        #     "Field166 string,"
        #     "Field167 string,"
        #     "Field168 string,"
        #     "Field169 string,"
        #     "Field170 string,"
        #     "Field171 string,"
        #     "Field172 string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExtKAMA"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== extkama ==================

        extkama_extra_filters = None

        if extkama_extra_filters is not None:
            extkama_extra_filters.append(date_filter)
            extkama_final_filters = " AND ".join(extkama_extra_filters)
        else:
            extkama_final_filters = date_filter

        extkama_selected_fields = [
            "BranchCode",
            "CustomerID_PEL",
            "CustomerName",
            "CustomerAddress",
            "CustomerCity",
            "CustomerTK",
            "CustomerAFM",
            "CustomerIDNumber",
            "CustomerBirthdate",
            "CustomerOccupation",
            "CustomerPhone",
            "StatementAddress",
            "StatementCity",
            "StatementTK",
            "GuarantorName",
            "GuarantorAddress",
            "GuarantorCity",
            "GuarantorTK",
            "GuarantorAFM",
            "GuarantorIDNumber",
            "GuarantorPhone",
            "GuarantorBirthdate",
            "GuarantorOccupation",
            "ContractNo",
            "ContractAmount",
            "ContractDate",
            "ContractUpdateDate",
            "AccountNo",
            "OpenAccountDate",
            "LegalFrame",
            "LegalFrameFlag",
            "AccountStatus",
            "PosostoEisforas",
            "InterestRate",
            "InterestRateType",
            "InterestRate_Basis",
            "InterestRateCategory",
            "DateLiksisEpitokiou",
            "InterestRateSpread",
            "InterestRateSpreadSign",
            "AmountApproved",
            "AmountDisbursed",
            "1stDisbursementDate",
            "TotalPayments",
            "TotalBalance",
            "Bucket0",
            "Bucket1",
            "Bucket2",
            "Bucket3",
            "Bucket4",
            "Bucket5",
            "Bucket6",
            "Bucket7+",
            "DelinquentAmount",
            "InstallmentAmount",
            "TotalInstallments",
            "DelinquentInstallments",
            "MaxBucketDate",
            "DaysInDelinquency",
            "LastPayDate",
            "LastPayAmount",
            "NextDueDate",
            "LoanDuration",
            "GracePeriodFlag",
            "GracePeriodEndDate",
            "TroposEksipiretisis",
            "ProductCode",
            "PrenotationFlag",
            "DateAnathesis",
            "FlagAnathesis",
            "AitiaEksairesis",
            "LastDisbursementDate",
            "BalanceBilling",
            "BalanceBucket0Billing",
            "CustomerSex",
            "BucketDME",
            "InstallmentsFrequency",
            "Proorismos",
            "DeferredCum",
            "ApplicationNo",
            "CustomerCategory",
            "DebitAccountNo",
            "Customer2Name",
            "Customer2Phone",
            "Customer3Name",
            "Customer3Phone",
            "Customer4Name",
            "Customer4Phone",
            "CommercialValue",
            "ObjectiveValue",
            "SeiraProsimeiosis",
            "PosoProsimeiosis",
            "KatigoriaAkinitou",
            "KladosOikonomikisDrastiriotitas",
            "Country",
            "Nationality",
            "Akinita_San_Collateral",
            "Akinita_Amounts",
            "YpoloipoTakseos_Akiniton",
            "Metoxes_San_Collateral",
            "Metoxes_Amounts",
            "YpoloipoTakseos_Metoxon",
            "Eggiiseis_San_Collateral",
            "Eggiiseis_Amounts",
            "YpoloipoTakseos_Eggiiseon",
            "Eggiitikes_San_Collateral",
            "Eggiitikes_Amounts",
            "YpoloipoTakseos_Eggiitikon",
            "Egg_Ell_Dimosiou_San_Collateral",
            "Egg_Ell_Dimosiou_Amounts",
            "YpoloipoTakseos_Egg_Ell_Dimosiou",
            "Loipes_Eggiiseis_San_Collateral",
            "Loipes_Eggiiseis_Amounts",
            "YpoloipoTakseos_Loipon_Eggiiseon",
            "Emporeumata_San_Collateral",
            "Emporeumata_Amounts",
            "YpoloipoTakseos_Emporeumaton",
            "Apothikografa_San_Collateral",
            "Apothikografa_Amounts",
            "YpoloipoTakseos_Apothikografon",
            "NeosTypos_San_Collateral",
            "NeosTypos_Amounts",
            "YpoloipoTakseos_NeouTypou",
            "LostInfo",
            "Next_Teliko_Epitokio",
            "Next_Eidos_Epitokiou",
            "Next_Epitokio_Vashs",
            "Next_Epitokio",
            "KodikosAkintou",
            "Bucket7",
            "Bucket8",
            "Bucket9",
            "Bucket10",
            "Bucket11",
            "Bucket12",
            "Bucket13",
            "DME_Diakanonismos",
            "DME_Rithmisi",
            "DME_Eggiisi_Ell_Dimosiou",
            "Add_ApplicationNo",
            "XrisiAkinitou",
            "AccStatus_Change_Date",
            "Monada_Diax_Enimerou_Pelati",
            "Perigrafi_Monadas_Diax_Enimerou_Pelati",
            "Monada_Diax_Anaggelmenou_Pelati",
            "Perigrafi_Monadas_Diax_Anaggelmenou_Pelati",
            "Ka8ysterimeno_Kefalaio",
            "Ka8ysterimenoi_Tokoi",
            "Ka8ysterimena_Eksoda",
            "Kefalaiopoihsh_Tokwn",
            "Epidotisielldimosiou",
            "DaysPastDueUnrealisedInterest",
            "CommercialFlag",
            "UnrealisedInterestBefore31032013",
            "EmporikhAgreementCode",
            "BucketNCKAG",
            "MinbucketNCKAG_KAMAB3",
            "LastRestructuringDate",
            "ExposureExpiredDate",
            "InitialExposureExpiredDate",
            "SigloProductCode",
            "AbhostAccount",
            "BU",
            "AbhostContract",
            "Field165",
            "Field166",
            "Field167",
            "Field168",
            "Field169",
            "Field170",
            "Field171",
            "Field172",
            "BusinessDate",
        ]

        extkama = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extkama",
            extkama_final_filters,
            extkama_selected_fields,
        )

        # ======================== result_df ========================

        primary_key_fields_id = [
            extkama.AccountNo,
            extkama.ContractNo,
            lit("extfile"),
        ]

        primary_key_fields_key = [
            extkama.AccountNo,
            extkama.ContractNo,
        ]

        extkama_df = extkama.withColumn(
            "ExtKAMAId",
            sha2(
                concat_ws(";", *primary_key_fields_id),
                256,
            ),
        ).withColumn(
            "ExtKAMAKey",
            concat_ws(";", *primary_key_fields_key),
        )

        result_df = extkama_df.select(
            extkama_df.ExtKAMAId.cast("string"),
            lit("extfile").alias("SourceSystem"),
            lit("-9999").cast("string").alias("PortfolioKey"),
            lit("Core Retail").cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            extkama_df.ExtKAMAKey.cast("string"),
            extkama.BranchCode.cast("string").alias("BranchCode"),
            extkama.AccountNo.cast("string").alias("ExposureKey"),
            extkama.ContractNo.cast("string").alias("AgreementKey"),
            extkama.CustomerID_PEL.cast("string").alias("CDI"),
            extkama.CustomerName.cast("string").alias("PartyName"),
            extkama.CustomerAddress.cast("string").alias("PartyAddress"),
            extkama.CustomerCity.cast("string").alias("PartyCity"),
            extkama.CustomerTK.cast("string").alias("PartyZipCode"),
            extkama.CustomerAFM.cast("string").alias("PartyTaxRegistrationNumber"),
            extkama.CustomerIDNumber.cast("string").alias("PartyIDNumber"),
            extkama.CustomerBirthdate.cast("string").alias("PartyBirthdate"),
            extkama.CustomerOccupation.cast("string").alias("PartyOccupation"),
            extkama.CustomerPhone.cast("string").alias("PartyPhone"),
            extkama.StatementAddress.cast("string").alias("StatementAddress"),
            extkama.StatementCity.cast("string").alias("StatementCity"),
            extkama.StatementTK.cast("string").alias("StatementTK"),
            extkama.GuarantorName.cast("string").alias("GuarantorName"),
            extkama.GuarantorAddress.cast("string").alias("GuarantorAddress"),
            extkama.GuarantorCity.cast("string").alias("GuarantorCity"),
            extkama.GuarantorTK.cast("string").alias("GuarantorTK"),
            extkama.GuarantorAFM.cast("string").alias("GuarantorAFM"),
            extkama.GuarantorIDNumber.cast("string").alias("GuarantorIDNumber"),
            extkama.GuarantorPhone.cast("string").alias("GuarantorPhone"),
            extkama.GuarantorBirthdate.cast("string").alias("GuarantorBirthdate"),
            extkama.GuarantorOccupation.cast("string").alias("GuarantorOccupation"),
            extkama.ContractAmount.cast("string").alias("AgreementAmount"),
            extkama.ContractDate.cast("string").alias("AgreementDate"),
            extkama.ContractUpdateDate.cast("string").alias("AgreementUpdateDate"),
            extkama.OpenAccountDate.cast("string").alias("OpenExposureDate"),
            extkama.LegalFrame.cast("string").alias("LegalFrame"),
            extkama.LegalFrameFlag.cast("string").alias("LegalFrameFlag"),
            extkama.AccountStatus.cast("string").alias("ExposureStatus"),
            extkama.PosostoEisforas.cast("string").alias("InstallmentPercentage"),
            extkama.InterestRate.cast("string").alias("InterestRate"),
            extkama.InterestRateType.cast("string").alias("InterestRateType"),
            extkama.InterestRate_Basis.cast("string").alias("InterestRateBasis"),
            extkama.InterestRateCategory.cast("string").alias("InterestRateCategory"),
            extkama.DateLiksisEpitokiou.cast("string").alias("InterestExpiredDate"),
            extkama.InterestRateSpread.cast("string").alias("InterestRateSpread"),
            extkama.InterestRateSpreadSign.cast("string").alias(
                "InterestRateSpreadSign"
            ),
            extkama.AmountApproved.cast("string").alias("AmountApproved"),
            extkama.AmountDisbursed.cast("string").alias("AmountDisbursed"),
            extkama["1stDisbursementDate"]
            .cast("string")
            .alias("FirstDisbursementDate"),
            extkama.TotalPayments.cast("string").alias("TotalPayments"),
            extkama.TotalBalance.cast("string").alias("TotalBalance"),
            extkama.Bucket0.cast("string").alias("Bucket0"),
            extkama.Bucket1.cast("string").alias("Bucket1"),
            extkama.Bucket2.cast("string").alias("Bucket2"),
            extkama.Bucket3.cast("string").alias("Bucket3"),
            extkama.Bucket4.cast("string").alias("Bucket4"),
            extkama.Bucket5.cast("string").alias("Bucket5"),
            extkama.Bucket6.cast("string").alias("Bucket6"),
            extkama["Bucket7+"].cast("string").alias("Bucket7Plus"),
            extkama.DelinquentAmount.cast("string").alias("DelinquentAmount"),
            extkama.InstallmentAmount.cast("string").alias("InstallmentAmount"),
            extkama.TotalInstallments.cast("string").alias("TotalInstallments"),
            extkama.DelinquentInstallments.cast("string").alias(
                "DelinquentInstallments"
            ),
            extkama.MaxBucketDate.cast("string").alias("BucketMaxDate"),
            extkama.DaysInDelinquency.cast("string").alias("DaysInDelinquency"),
            extkama.LastPayDate.cast("string").alias("LastPayDate"),
            extkama.LastPayAmount.cast("string").alias("LastPayAmount"),
            extkama.NextDueDate.cast("string").alias("NextDueDate"),
            extkama.LoanDuration.cast("string").alias("ExposureDuration"),
            extkama.GracePeriodFlag.cast("string").alias("GracePeriodFlag"),
            extkama.GracePeriodEndDate.cast("string").alias("GracePeriodEndDate"),
            extkama.TroposEksipiretisis.cast("string").alias("ServiceWay"),
            extkama.ProductCode.cast("string").alias("ProductCode"),
            extkama.PrenotationFlag.cast("string").alias("PrenotationFlag"),
            extkama.DateAnathesis.cast("string").alias("AssignmentDate"),
            extkama.FlagAnathesis.cast("string").alias("AssignmentFlag"),
            extkama.AitiaEksairesis.cast("string").alias("ExceptionReason"),
            extkama.LastDisbursementDate.cast("string").alias("DisbursementLastDate"),
            extkama.BalanceBilling.cast("string").alias("BalanceBilling"),
            extkama.BalanceBucket0Billing.cast("string").alias("BalanceBucket0Billing"),
            extkama.CustomerSex.cast("string").alias("PartyGender"),
            extkama.BucketDME.cast("string").alias("BucketDME"),
            extkama.InstallmentsFrequency.cast("string").alias("InstallmentsFrequency"),
            extkama.Proorismos.cast("string").alias("Proorismos"),
            extkama.DeferredCum.cast("string").alias("DeferredCum"),
            extkama.ApplicationNo.cast("string").alias("ApplicationCode"),
            extkama.CustomerCategory.cast("string").alias("PartyCategory"),
            extkama.DebitAccountNo.cast("string").alias("DebitExposureKey"),
            extkama.Customer2Name.cast("string").alias("Party2Name"),
            extkama.Customer2Phone.cast("string").alias("Party2Phone"),
            extkama.Customer3Name.cast("string").alias("Party3Name"),
            extkama.Customer3Phone.cast("string").alias("Party3Phone"),
            extkama.Customer4Name.cast("string").alias("Party4Name"),
            extkama.Customer4Phone.cast("string").alias("Party4Phone"),
            extkama.CommercialValue.cast("string").alias("CommercialValue"),
            extkama.ObjectiveValue.cast("string").alias("ObjectiveValue"),
            extkama.SeiraProsimeiosis.cast("string").alias("LienOrder"),
            extkama.PosoProsimeiosis.cast("string").alias("LienAmount"),
            extkama.KatigoriaAkinitou.cast("string").alias("RealEstateCategory"),
            extkama.KladosOikonomikisDrastiriotitas.cast("string").alias("NaceCode"),
            extkama.Country.cast("string").alias("Country"),
            extkama.Nationality.cast("string").alias("Nationality"),
            extkama.Akinita_San_Collateral.cast("string").alias("RealEstateCollateral"),
            extkama.Akinita_Amounts.cast("string").alias("RealEstateAmount"),
            extkama.YpoloipoTakseos_Akiniton.cast("string").alias(
                "OnBalanceRealEstateAmount"
            ),
            extkama.Metoxes_San_Collateral.cast("string").alias("SharesCollateral"),
            extkama.Metoxes_Amounts.cast("string").alias("SharesAmount"),
            extkama.YpoloipoTakseos_Metoxon.cast("string").alias(
                "OnBalanceSharesAmount"
            ),
            extkama.Eggiiseis_San_Collateral.cast("string").alias(
                "LetterOfCreditCollateral"
            ),
            extkama.Eggiiseis_Amounts.cast("string").alias("LetterOfCreditAmount"),
            extkama.YpoloipoTakseos_Eggiiseon.cast("string").alias(
                "OnBalanceLetterOfCreditAmount"
            ),
            extkama.Eggiitikes_San_Collateral.cast("string").alias(
                "LetterOfGuranteeCollateral"
            ),
            extkama.Eggiitikes_Amounts.cast("string").alias("LetterOfGuranteeAmounts"),
            extkama.YpoloipoTakseos_Eggiitikon.cast("string").alias(
                "OnBalanceLetterOfGuaranteeAmount"
            ),
            extkama.Egg_Ell_Dimosiou_San_Collateral.cast("string").alias(
                "LetterOfPublicGuaranteeCollateral"
            ),
            extkama.Egg_Ell_Dimosiou_Amounts.cast("string").alias(
                "LetterOfPublicGuaranteeAmount"
            ),
            extkama.YpoloipoTakseos_Egg_Ell_Dimosiou.cast("string").alias(
                "OnBalanceLetterOfPublicGuaranteeAmount"
            ),
            extkama.Loipes_Eggiiseis_San_Collateral.cast("string").alias(
                "OtherOfGuaranteesCollateral"
            ),
            extkama.Loipes_Eggiiseis_Amounts.cast("string").alias(
                "OtherOfGuaranteesAmount"
            ),
            extkama.YpoloipoTakseos_Loipon_Eggiiseon.cast("string").alias(
                "OnBalanceOtherOfGuaranteesAmount"
            ),
            extkama.Emporeumata_San_Collateral.cast("string").alias("GoodsCollateral"),
            extkama.Emporeumata_Amounts.cast("string").alias("GoodsAmount"),
            extkama.YpoloipoTakseos_Emporeumaton.cast("string").alias(
                "OnBalanceGoodsAmount"
            ),
            extkama.Apothikografa_San_Collateral.cast("string").alias(
                "ArchiveFilesCollateral"
            ),
            extkama.Apothikografa_Amounts.cast("string").alias("ArchiveFilesAmount"),
            extkama.YpoloipoTakseos_Apothikografon.cast("string").alias(
                "OnBalanceArchiveFilesAmount"
            ),
            extkama.NeosTypos_San_Collateral.cast("string").alias("NewTypeCollateral"),
            extkama.NeosTypos_Amounts.cast("string").alias("NewTypeAmount"),
            extkama.YpoloipoTakseos_NeouTypou.cast("string").alias(
                "OnBalanceNewTypeAmount"
            ),
            extkama.LostInfo.cast("string").alias("ExposureInfo"),
            extkama.Next_Teliko_Epitokio.cast("string").alias("NextFinalInterest"),
            extkama.Next_Eidos_Epitokiou.cast("string").alias("NextInerestType"),
            extkama.Next_Epitokio_Vashs.cast("string").alias("NextBaseRate"),
            extkama.Next_Epitokio.cast("string").alias("NextInterest"),
            extkama.KodikosAkintou.cast("string").alias("RealEstateKey"),
            extkama.Bucket7.cast("string").alias("Bucket7"),
            extkama.Bucket8.cast("string").alias("Bucket8"),
            extkama.Bucket9.cast("string").alias("Bucket9"),
            extkama.Bucket10.cast("string").alias("Bucket10"),
            extkama.Bucket11.cast("string").alias("Bucket11"),
            extkama.Bucket12.cast("string").alias("Bucket12"),
            extkama.Bucket13.cast("string").alias("Bucket13"),
            extkama.DME_Diakanonismos.cast("string").alias("DMESettlement"),
            extkama.DME_Rithmisi.cast("string").alias("DMERestructuring"),
            extkama.DME_Eggiisi_Ell_Dimosiou.cast("string").alias(
                "DMELetterOfPublicGurantee"
            ),
            extkama.Add_ApplicationNo.cast("string").alias("AddApplicationKey"),
            extkama.XrisiAkinitou.cast("string").alias("RealEstateUsage"),
            extkama.AccStatus_Change_Date.cast("string").alias(
                "ExposureStatusChangeDate"
            ),
            extkama.Monada_Diax_Enimerou_Pelati.cast("string").alias(
                "PerformingPartyUnitCode"
            ),
            extkama.Perigrafi_Monadas_Diax_Enimerou_Pelati.cast("string").alias(
                "PerformingPartyUnit"
            ),
            extkama.Monada_Diax_Anaggelmenou_Pelati.cast("string").alias(
                "AnnouncedPartyUnitCode"
            ),
            extkama.Perigrafi_Monadas_Diax_Anaggelmenou_Pelati.cast("string").alias(
                "AnnouncedPartyUnit"
            ),
            extkama.Ka8ysterimeno_Kefalaio.cast("string").alias("DueCapital"),
            extkama.Ka8ysterimenoi_Tokoi.cast("string").alias("DueInterest"),
            extkama.Ka8ysterimena_Eksoda.cast("string").alias("DueExpense"),
            extkama.Kefalaiopoihsh_Tokwn.cast("string").alias(
                "CapitalisationOfInterest"
            ),
            extkama["Epidotisielldimosiou"].cast("string").alias("SubsidyByPublic"),
            extkama.DaysPastDueUnrealisedInterest.cast("string").alias(
                "DaysPastDueUnrealisedInterest"
            ),
            extkama.CommercialFlag.cast("string").alias("CommercialFlag"),
            extkama.UnrealisedInterestBefore31032013.cast("string").alias(
                "UnrealisedInterestBefore31032013"
            ),
            extkama.EmporikhAgreementCode.cast("string").alias("EmporikhAgreementCode"),
            extkama["BucketNCKAG"].cast("string").alias("BucketNCKAG"),
            extkama["MinbucketNCKAG_KAMAB3"].cast("string").alias("MinBucketNCKAG"),
            extkama.LastRestructuringDate.cast("string").alias("LastRestructuringDate"),
            extkama.ExposureExpiredDate.cast("string").alias("ExposureExpiredDate"),
            extkama.InitialExposureExpiredDate.cast("string").alias(
                "InitialExposureExpiredDate"
            ),
            extkama["SigloProductCode"].cast("string").alias("SigloProductCode"),
            extkama["AbhostAccount"].cast("string").alias("AbhostExposure"),
            extkama.BU.cast("string").alias("BU"),
            extkama["AbhostContract"].cast("string").alias("AbhostAgreement"),
            extkama.Field165.cast("string").alias("Field165"),
            extkama.Field166.cast("string").alias("Field166"),
            extkama.Field167.cast("string").alias("Field167"),
            extkama.Field168.cast("string").alias("Field168"),
            extkama.Field169.cast("string").alias("Field169"),
            extkama.Field170.cast("string").alias("Field170"),
            extkama.Field171.cast("string").alias("Field171"),
            extkama.Field172.cast("string").alias("Field172"),
            extkama.BusinessDate.cast("date").alias("BusinessDate"),
        )
        # ========================== Append ==========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_ext_kama_extfile(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)